In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from imblearn.over_sampling import SMOTE


# ============================================================
# SIMULATION OF A DATA SET WITH BINARY TARGET VARIABLE
# WITH CLASSES "majority" AND "minority"
# ============================================================
np.random.seed(236683) 
n_majority = 1000
n_minority = 200

df_majority = pd.DataFrame({
    "x1": np.random.normal(loc=0, size=n_majority),
    "x2": np.random.normal(loc=1, size=n_majority),
    "x3": np.random.binomial(1, 0.3, size=n_majority),
    "Class": "majority"
})

df_minority = pd.DataFrame({
    "x1": np.random.normal(loc=1, size=n_minority),
    "x2": np.random.normal(loc=2, size=n_minority),
    "x3": np.random.binomial(1, 0.6, size=n_minority),
    "Class": "minority"
})

df = pd.concat([df_majority, df_minority], ignore_index=True)
df["Class"] = pd.Categorical(df["Class"], categories=["majority", "minority"])

print(df["Class"].value_counts().sort_index())


# ============================================================
# RANDOM SUBSAMPLING OF MAJORITY CLASS
# ============================================================
np.random.seed(293024)

maj = df[df["Class"] == "majority"].copy()
mino = df[df["Class"] == "minority"].copy()
n_min = len(mino)

df_rand_2to1 = pd.concat([
    maj.sample(n=2 * n_min, replace=False, random_state=293024),
    mino
], ignore_index=True)

df_rand_3to1 = pd.concat([
    maj.sample(n=3 * n_min, replace=False, random_state=293025),
    mino
], ignore_index=True)

df_rand_4to1 = pd.concat([
    maj.sample(n=4 * n_min, replace=False, random_state=293026),
    mino
], ignore_index=True)

# shuffling rows
df_rand_2to1 = df_rand_2to1.sample(frac=1, random_state=293024).reset_index(drop=True)
df_rand_3to1 = df_rand_3to1.sample(frac=1, random_state=293024).reset_index(drop=True)
df_rand_4to1 = df_rand_4to1.sample(frac=1, random_state=293024).reset_index(drop=True)

# checking class balance
print("\nRandom 2:1")
print(df_rand_2to1["Class"].value_counts().sort_index())

print("\nRandom 3:1")
print(df_rand_3to1["Class"].value_counts().sort_index())

print("\nRandom 4:1")
print(df_rand_4to1["Class"].value_counts().sort_index())


# ============================================================
# PROPENSITY SCORE MATCHING SUBSAMPLING OF MAJORITY CLASS
# ============================================================
df_ps = df.copy()
df_ps["mino"] = np.where(df_ps["Class"] == "minority", 1, 0)

X_ps = df_ps[["x1", "x2", "x3"]]
y_ps = df_ps["mino"]

# Estimate propensity scores
ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(X_ps, y_ps)
df_ps["propensity_score"] = ps_model.predict_proba(X_ps)[:, 1]


def propensity_score_match(data, ratio=2):
    minority = data[data["mino"] == 1].copy()
    majority = data[data["mino"] == 0].copy()

    # nearest neighbors on propensity score
    nn = NearestNeighbors(n_neighbors=ratio)
    nn.fit(majority[["propensity_score"]])

    distances, indices = nn.kneighbors(minority[["propensity_score"]])

    matched_majority_idx = majority.iloc[indices.flatten()].index
    matched_majority = data.loc[matched_majority_idx, ["x1", "x2", "x3", "Class"]]
    matched_minority = minority[["x1", "x2", "x3", "Class"]]

    matched_df = pd.concat([matched_majority, matched_minority], ignore_index=True)
    matched_df = matched_df.sample(frac=1, random_state=293024).reset_index(drop=True)
    return matched_df


df_match_2to1 = propensity_score_match(df_ps, ratio=2)
df_match_3to1 = propensity_score_match(df_ps, ratio=3)
df_match_4to1 = propensity_score_match(df_ps, ratio=4)

print("\nMatched 2:1")
print(df_match_2to1["Class"].value_counts().sort_index())

print("\nMatched 3:1")
print(df_match_3to1["Class"].value_counts().sort_index())

print("\nMatched 4:1")
print(df_match_4to1["Class"].value_counts().sort_index())


# ============================================================
# OVERSAMPLING MINORITY CLASS USING SMOTE
# ============================================================
X = df[["x1", "x2", "x3"]]
y = df["Class"]

smote = SMOTE(sampling_strategy=0.6, k_neighbors=5, random_state=304533)
X_smote, y_smote = smote.fit_resample(X, y)

df_smote = pd.DataFrame(X_smote, columns=["x1", "x2", "x3"])
df_smote["Class"] = y_smote

print("\nSMOTE class balance")
print(df_smote["Class"].value_counts().sort_index())

print("\nSMOTE x3 values")
print(df_smote["x3"].value_counts().sort_index())
print("\nUnique x3 values after SMOTE:")
print(np.sort(df_smote["x3"].unique()))